# Experiment 3.13: SGD→Muon Hybrid Fine-Tuning Strategy

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/SGD_THEN_MUON_hybrid/run_experiment.py`

This notebook is the analysis/presentation layer for the same deep-linear switch-point sweep implemented in the script. The script now owns the experiment logic via `run_experiment()`, while this notebook imports that code, runs the default configuration, and analyzes the returned structured results.

## Truthful scope

- **Model/task:** 4-layer deep linear network in a controlled toy setting.
- **Perturbation protocol:** pre-train on one fixed target matrix, then randomly replace 20% of target entries before fine-tuning.
- **Measured outputs:** fine-tuning loss trajectories, final loss after 200 steps, and steps required to reach 50% of the initial fine-tuning loss.
- **What this notebook does not claim:** it does **not** provide a mechanistic proof of a spectral/gauge phase transition, a Hessian-level analysis, or a realistic distribution-shift study.


## Measured claims and framing

The question here is deliberately narrow: **how does the switch point `S` in a toy SGD→Muon fine-tuning schedule affect early speed and final loss?**

We distinguish two different notions of "best":

1. **Best overall by mean final loss** across all switch points, including pure Muon (`S=0`) and pure SGD (`S=200`).
2. **Best non-pure hybrid** among the intermediate switch points `0 < S < 200`.

That distinction matters because a non-pure hybrid can be **Pareto-competitive** without being globally best on the final-loss objective.


In [ ]:
import importlib.util
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

try:
    from IPython.display import Markdown, display

    def show_markdown(text):
        display(Markdown(text))
except ImportError:
    def display(obj):
        print(obj)

    def show_markdown(text):
        print(text)


def find_repo_root(start=None):
    start_path = Path(start or os.getcwd()).resolve()
    relative_script = Path('experiments/Tier_1_Core_Mechanism_Tests/SGD_THEN_MUON_hybrid/run_experiment.py')
    for candidate in [start_path, *start_path.parents]:
        if (candidate / relative_script).exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate repo root containing experiments/Tier_1_Core_Mechanism_Tests/SGD_THEN_MUON_hybrid/run_experiment.py'
    )


def load_module_from_path(module_name, module_path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def show_dataframe(df, title=None, index=False):
    if title:
        print(title)
    try:
        display(df if index else df.reset_index(drop=True))
    except Exception:
        if index:
            print(df.to_string())
        else:
            print(df.reset_index(drop=True).to_string(index=False))


## Reproducibility and execution plan

This notebook intentionally avoids duplicating the training loop. Instead it:

1. Locates the repository root from the current working directory without relying on `__file__`.
2. Imports `run_experiment.py` as a module.
3. Runs `run_experiment(verbose=False)` with the script defaults.
4. Builds tables and figures from the returned structured results.

For command-line reproduction, the paired script entrypoint is:

```bash
python experiments/Tier_1_Core_Mechanism_Tests/SGD_THEN_MUON_hybrid/run_experiment.py
```


In [ ]:
REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'SGD_THEN_MUON_hybrid'
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'

experiment = load_module_from_path('sgd_then_muon_hybrid_experiment', SCRIPT_PATH)

t0 = time.perf_counter()
results = experiment.run_experiment(verbose=False)
runtime_seconds = time.perf_counter() - t0

config = results['config']
summary_df = pd.DataFrame(results['summary_table']).sort_values('S').reset_index(drop=True)
ranking_df = pd.DataFrame(results['ranking_by_final_loss'])

best_overall_S = results['best_overall_S']
best_nonpure_S = results['best_nonpure_hybrid_S']
pure_muon_S = 0
pure_sgd_S = config['finetune_steps']
tracked_switches = []
for S in [pure_muon_S, best_nonpure_S, pure_sgd_S]:
    if S is not None and S not in tracked_switches:
        tracked_switches.append(S)

print(f'Repo root: {REPO_ROOT}')
print(f'Script path: {SCRIPT_PATH}')
print(f'Notebook runtime for run_experiment(): {runtime_seconds:.3f} s')
print(f'Seeds used: {results["seeds"]}')
print(f'Best overall by mean final loss: S={best_overall_S}')
print(f'Best non-pure hybrid by mean final loss: S={best_nonpure_S}')


## Runtime, configuration, and fixed-data logging

The tables below record the exact default configuration, the seed set, and basic fixed-data statistics returned by the script. This is the minimum information needed to make the notebook self-explanatory and reproducible.


In [ ]:
runtime_df = pd.DataFrame([
    {'field': 'counterpart_script', 'value': str(SCRIPT_PATH.relative_to(REPO_ROOT))},
    {'field': 'runtime_seconds', 'value': runtime_seconds},
    {'field': 'num_seeds', 'value': config['num_seeds']},
    {'field': 'seeds', 'value': ', '.join(str(s) for s in results['seeds'])},
    {'field': 'best_overall_S', 'value': best_overall_S},
    {'field': 'best_nonpure_hybrid_S', 'value': best_nonpure_S},
])

config_df = pd.DataFrame([
    {'parameter': key, 'value': value}
    for key, value in config.items()
    if key not in {'switch_points', 'snapshot_steps'}
])
config_df = pd.concat([
    config_df,
    pd.DataFrame([
        {'parameter': 'switch_points', 'value': config['switch_points']},
        {'parameter': 'snapshot_steps', 'value': results['snapshot_steps']},
    ])
], ignore_index=True)

fixed = results['fixed_data']
fixed_data_df = pd.DataFrame([
    {'field': 'X_shape', 'value': fixed['X_shape']},
    {'field': 'target_shape', 'value': fixed['target_shape']},
    {'field': 'X_mean', 'value': fixed['X_mean']},
    {'field': 'X_std', 'value': fixed['X_std']},
    {'field': 'target_mean', 'value': fixed['target_mean']},
    {'field': 'target_std', 'value': fixed['target_std']},
    {'field': 'target_fro_norm', 'value': fixed['target_fro_norm']},
    {'field': 'top_5_target_singular_values', 'value': np.round(fixed['target_singular_values'][:5], 6).tolist()},
])

show_dataframe(runtime_df, 'Runtime / identity log')
show_dataframe(config_df, 'Default configuration used')
show_dataframe(fixed_data_df, 'Fixed-data summary returned by the script')


## Aggregate switch-point results

The next tables summarize the script outputs over seeds. The first is the full switch-point sweep. The second isolates the **best overall by final loss** and the **best non-pure hybrid**, so the notebook does not conflate those two identities.


In [ ]:
summary_view = summary_df[[
    'S', 'description',
    'mean_final_loss', 'std_final_loss', 'ci95_final_loss',
    'mean_half_steps', 'std_half_steps', 'ci95_half_steps'
]].copy()
show_dataframe(summary_view, 'Full switch-point summary')

show_dataframe(
    ranking_df[[
        'rank', 'S', 'description', 'mean_final_loss',
        'mean_half_steps', 'is_best_overall', 'is_best_nonpure_hybrid'
    ]],
    'Ranking by mean final loss'
)

best_tables = []
for label, S in [
    ('Best overall by mean final loss', best_overall_S),
    ('Best non-pure hybrid by mean final loss', best_nonpure_S),
    ('Pure Muon baseline', pure_muon_S),
    ('Pure SGD baseline', pure_sgd_S),
]:
    if S is None:
        continue
    row = summary_df.loc[summary_df['S'] == S].iloc[0].to_dict()
    row = {
        'identity': label,
        'S': row['S'],
        'description': row['description'],
        'mean_final_loss': row['mean_final_loss'],
        'ci95_final_loss': row['ci95_final_loss'],
        'mean_half_steps': row['mean_half_steps'],
        'ci95_half_steps': row['ci95_half_steps'],
    }
    best_tables.append(row)

best_df = pd.DataFrame(best_tables)
show_dataframe(best_df, 'Best-overall / best-nonpure / baseline comparison')


In [ ]:
best_overall_desc = results['best_overall_description']
best_nonpure_desc = results['best_nonpure_hybrid_description']
pure_muon_final = results['per_switch'][pure_muon_S]['mean_final_loss']
pure_sgd_final = results['per_switch'][pure_sgd_S]['mean_final_loss']

if best_nonpure_S is None:
    interpretation = f"""
### Aggregate interpretation

- **Best overall by mean final loss:** `S={best_overall_S}` ({best_overall_desc}).
- No non-pure hybrid switch points were supplied, so there is no intermediate hybrid identity to compare.
- The toy perturbation protocol still supports baseline comparison, but not a hybrid-selection claim.
"""
else:
    best_nonpure_final = results['per_switch'][best_nonpure_S]['mean_final_loss']
    ratio = best_nonpure_final / (pure_muon_final + 1e-12)
    interpretation = f"""
### Aggregate interpretation

- **Best overall by mean final loss:** `S={best_overall_S}` ({best_overall_desc}).
- **Best non-pure hybrid by mean final loss:** `S={best_nonpure_S}` ({best_nonpure_desc}).
- Pure Muon mean final loss is `{pure_muon_final:.6f}` and pure SGD mean final loss is `{pure_sgd_final:.6f}`.
- The best non-pure hybrid has mean final loss `{best_nonpure_final:.6f}`, i.e. `{ratio:.4f}×` the pure-Muon mean final loss.
- This supports a trade-off / Pareto-style reading rather than a blanket claim that a hybrid beats pure Muon on the final-loss objective.
"""

show_markdown(interpretation)


## Figure 1 — Fine-tuning loss trajectories

This figure compares the seed-averaged loss trajectories for three reference settings:

- pure Muon (`S=0`)
- the best non-pure hybrid (if one exists)
- pure SGD (`S=200`)

Bands show **95% confidence intervals across seeds** from the returned script results.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
color_map = {
    pure_muon_S: '#1f77b4',
    pure_sgd_S: '#2ca02c',
}
if best_nonpure_S is not None:
    color_map[best_nonpure_S] = '#ff7f0e'

for S in tracked_switches:
    label = results['per_switch'][S]['description']
    if S == best_nonpure_S and S not in {pure_muon_S, pure_sgd_S}:
        label = f'Best non-pure hybrid: {label}'
    elif S == pure_muon_S:
        label = 'Pure Muon'
    elif S == pure_sgd_S:
        label = 'Pure SGD'

    mean_curve = results['per_switch'][S]['mean_loss_curve']
    ci_curve = results['per_switch'][S]['ci95_loss_curve']
    steps = np.arange(len(mean_curve))

    ax.plot(steps, mean_curve, linewidth=2.2, label=label, color=color_map.get(S, None))
    ax.fill_between(
        steps,
        mean_curve - ci_curve,
        mean_curve + ci_curve,
        alpha=0.18,
        color=color_map.get(S, None),
    )

if best_nonpure_S is not None and best_nonpure_S not in {pure_muon_S, pure_sgd_S}:
    ax.axvline(
        best_nonpure_S,
        color='black',
        linestyle='--',
        linewidth=1.2,
        alpha=0.7,
        label=f'Hybrid switch at S={best_nonpure_S}'
    )

ax.set_title('Fine-tuning loss trajectories (mean ± 95% CI over seeds)')
ax.set_xlabel('Trajectory index (0 = initial fine-tuning loss, 200 = final post-update loss)')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
if best_nonpure_S is None:
    text = f"""
### Figure 1 interpretation

- Only the pure baselines are available in this run.
- Pure Muon ends with mean final loss `{pure_muon_final:.6f}` and pure SGD ends with `{pure_sgd_final:.6f}`.
- The figure is therefore a baseline comparison, not a hybrid trade-off analysis.
"""
else:
    best_nonpure_final = results['per_switch'][best_nonpure_S]['mean_final_loss']
    pure_muon_half = results['per_switch'][pure_muon_S]['mean_half_steps']
    best_nonpure_half = results['per_switch'][best_nonpure_S]['mean_half_steps']
    pure_sgd_half = results['per_switch'][pure_sgd_S]['mean_half_steps']
    text = f"""
### Figure 1 interpretation

- Pure Muon (`S=0`) achieves the lowest mean final loss in this run: `{pure_muon_final:.6f}`.
- The best non-pure hybrid is `S={best_nonpure_S}`, with mean final loss `{best_nonpure_final:.6f}`.
- Mean half-loss steps are `{pure_muon_half:.1f}` for pure Muon, `{best_nonpure_half:.1f}` for the best non-pure hybrid, and `{pure_sgd_half:.1f}` for pure SGD.
- The figure therefore supports a **speed/quality trade-off** interpretation: the non-pure hybrid can be faster to the 50%-loss threshold while remaining close to pure Muon in final loss, but pure Muon still wins the final-loss objective.
"""
show_markdown(text)


## Figure 2 — Mean final loss versus switch point

This figure treats **mean final loss** as the primary quality objective and shows uncertainty across seeds via 95% confidence intervals.


In [ ]:
switch_points = summary_df['S'].to_numpy()
mean_finals = summary_df['mean_final_loss'].to_numpy()
ci_finals = summary_df['ci95_final_loss'].to_numpy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(switch_points, mean_finals, yerr=ci_finals, fmt='o-', capsize=4, linewidth=2)
ax.axvline(best_overall_S, color='#1f77b4', linestyle='--', alpha=0.7, label=f'Best overall S={best_overall_S}')
if best_nonpure_S is not None:
    ax.axvline(best_nonpure_S, color='#ff7f0e', linestyle=':', alpha=0.9, label=f'Best non-pure S={best_nonpure_S}')
ax.set_title('Mean final loss after 200 fine-tuning steps')
ax.set_xlabel('Switch point S (SGD steps before switching to Muon)')
ax.set_ylabel('Mean final loss')
ax.grid(True, alpha=0.3)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
if best_nonpure_S is None:
    text = f"""
### Figure 2 interpretation

- The best overall switch point by mean final loss is `S={best_overall_S}`.
- Because no intermediate hybrid setting is available, the plot only compares the pure baselines.
"""
else:
    best_overall_final = results['per_switch'][best_overall_S]['mean_final_loss']
    best_nonpure_final = results['per_switch'][best_nonpure_S]['mean_final_loss']
    delta = best_nonpure_final - best_overall_final
    text = f"""
### Figure 2 interpretation

- **Best overall by mean final loss:** `S={best_overall_S}` with `{best_overall_final:.6f}`.
- **Best non-pure hybrid:** `S={best_nonpure_S}` with `{best_nonpure_final:.6f}`.
- The absolute final-loss gap between those two identities is `{delta:.6f}`.
- This is the clearest place to avoid overclaiming: if `S=0` is best overall, then the notebook should say so explicitly rather than describing the best non-pure hybrid as globally optimal.
"""
show_markdown(text)


## Figure 3 — Steps to 50% of the initial fine-tuning loss

This figure focuses on a simple convergence-speed proxy: the first trajectory index at which loss falls to 50% of the initial fine-tuning loss.


In [ ]:
mean_halves = summary_df['mean_half_steps'].to_numpy()
ci_halves = summary_df['ci95_half_steps'].to_numpy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(
    switch_points,
    mean_halves,
    yerr=ci_halves,
    fmt='o-',
    capsize=4,
    linewidth=2,
    color='#9467bd'
)
ax.axvline(
    results['fastest_half_step_S'],
    color='black',
    linestyle='--',
    alpha=0.7,
    label=f'Fastest mean half-step S={results["fastest_half_step_S"]}'
)
if best_nonpure_S is not None:
    ax.axvline(best_nonpure_S, color='#ff7f0e', linestyle=':', alpha=0.9, label=f'Best non-pure S={best_nonpure_S}')
ax.set_title('Mean steps to 50% of initial fine-tuning loss')
ax.set_xlabel('Switch point S (SGD steps before switching to Muon)')
ax.set_ylabel('Mean half-loss step')
ax.grid(True, alpha=0.3)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
fastest_S = results['fastest_half_step_S']
fastest_desc = results['fastest_half_step_description']
fastest_half = results['per_switch'][fastest_S]['mean_half_steps']

if best_nonpure_S is None:
    text = f"""
### Figure 3 interpretation

- The fastest setting to the half-loss threshold is `S={fastest_S}` ({fastest_desc}) with mean `{fastest_half:.1f}` steps.
- Without intermediate hybrid settings, this remains a pure-baseline speed comparison.
"""
else:
    pure_muon_half = results['per_switch'][pure_muon_S]['mean_half_steps']
    best_nonpure_half = results['per_switch'][best_nonpure_S]['mean_half_steps']
    text = f"""
### Figure 3 interpretation

- The fastest setting to the half-loss threshold is `S={fastest_S}` ({fastest_desc}) with mean `{fastest_half:.1f}` steps.
- Pure Muon reaches the threshold in `{pure_muon_half:.1f}` mean steps.
- The best non-pure hybrid reaches the threshold in `{best_nonpure_half:.1f}` mean steps.
- This speed proxy is intentionally modest: it captures early descent timing, not overall optimization quality or mechanism.
"""
show_markdown(text)


## Heuristic T1/T2/T3/T4 checks

These checks are retained from the paired script, but they are now labeled more carefully. They are **heuristic comparisons of sample means across seeds**, not formal hypothesis tests.


In [ ]:
tests_df = pd.DataFrame([
    {
        'test': name,
        'question': results['tests'][name]['question'],
        'passed': results['tests'][name]['passed'],
        'details': results['tests'][name]['details'],
    }
    for name in ['T1', 'T2', 'T3', 'T4']
])
show_dataframe(tests_df, 'Heuristic T1/T2/T3/T4 outcomes')
print('Caveat:', results['tests']['caveat'])
print('Total passed:', results['tests']['total_passed'], '/ 4')


In [ ]:
passed_tests = [name for name in ['T1', 'T2', 'T3', 'T4'] if results['tests'][name]['passed']]
failed_tests = [name for name in ['T1', 'T2', 'T3', 'T4'] if not results['tests'][name]['passed']]

text = f"""
### Heuristic-check interpretation

- Passed checks: `{passed_tests}`.
- Failed checks: `{failed_tests}`.
- Because these are based on mean summaries over only `{config['num_seeds']}` seeds, they should be read as compact diagnostic criteria rather than formal statistical evidence.
- Their main value here is descriptive: they summarize whether the toy run exhibits the intended speed/quality trade-off pattern.
"""
show_markdown(text)


## Final conclusion

The conclusion below is generated from the actual returned results so that the notebook states the correct identities for **best overall** and **best non-pure hybrid** without conflating them.


In [ ]:
best_overall_row = summary_df.loc[summary_df['S'] == best_overall_S].iloc[0]

if best_nonpure_S is None:
    conclusion = f"""
### Conclusion

- **Best overall by mean final loss:** `S={best_overall_S}` ({best_overall_row['description']}).
- No intermediate hybrid switch point was available, so no best non-pure hybrid can be reported.
- The results should be interpreted as a toy deep-linear baseline comparison under a target-perturbation protocol.
- They do **not** by themselves establish a mechanistic explanation or a realistic distribution-shift result.
"""
else:
    best_nonpure_row = summary_df.loc[summary_df['S'] == best_nonpure_S].iloc[0]
    conclusion = f"""
### Conclusion

- **Best overall by mean final loss:** `S={best_overall_S}` ({best_overall_row['description']}) with mean final loss `{best_overall_row['mean_final_loss']:.6f}`.
- **Best non-pure hybrid by mean final loss:** `S={best_nonpure_S}` ({best_nonpure_row['description']}) with mean final loss `{best_nonpure_row['mean_final_loss']:.6f}`.
- In this default run, the correct reading is: **pure Muon wins the final-loss objective, while the best non-pure hybrid is the main Pareto-competitive setting to inspect for a speed/final-loss trade-off**.
- The perturbation remains a toy target-entry replacement protocol, so any broader spectral or gauge interpretation should be treated as motivation rather than demonstrated mechanism.
- For stricter follow-up validation, the next step would be to re-run with more seeds and/or add explicit statistical and mechanistic instrumentation rather than strengthening the current narrative.
"""

show_markdown(conclusion)
print('Script verdict:', results['verdict'])
